# NLP Mastery Journey — Module 1: Data Acquisition

Welcome! This is Lesson 1 of your path from **zero to advanced NLP**. Before you can clean, tokenize, model, or fine-tune anything, you need **data** — and real-world data almost never arrives as a neat CSV. It shows up as scraped HTML, PDFs, scanned images, podcasts, API JSON blobs, and video files.

### What this notebook teaches
| # | Source | You will learn |
|---|--------|-----------------|
| 1 | Local files (CSV/TSV/JSON/Excel) | `pandas`, the standard `csv` module, and when to use each |
| 2 | Public NLP datasets | `datasets` (Hugging Face), `sklearn`, `nltk` corpora |
| 3 | Web scraping | `requests` + `BeautifulSoup`, pagination, ethics/`robots.txt` |
| 4 | REST APIs | auth, pagination, rate limits, a reusable fetch template |
| 5 | Text-bearing files | `.txt`, `.docx`, `.pdf` (two libraries compared) |
| 6 | Images (OCR) | `pytesseract`, image pre-processing |
| 7 | Audio | Whisper & `SpeechRecognition` for transcription |
| 8 | Video | extracting the audio track, then transcribing it |
| 9 | Consolidation | turning everything into one clean corpus format |
| 10 | Production practices | retries, caching, logging, rate-limiting, legal notes |

### How to use this notebook
- **Every code cell is commented line-by-line** — read the comments, not just the code.
- **"🔀 Alternatives" callouts** tell you what else exists and *when* you'd reach for it instead.
- **"📋 Copy-paste template" cells** are deliberately written as generic, reusable functions. In your next project, you can lift these almost unchanged — just swap the URL/path/column names.
- This notebook is built to **run inside a sandbox with no internet access**, so network cells are written correctly but will raise a connection error here. Run them on your own machine (or any environment with internet) — that's the whole point of a template: it works wherever you paste it.
- Cells that need an install are marked; uncomment the `pip install` line the first time you run them.

Let's build your data-acquisition toolkit.


## 0. Setup

We install everything up front so you're not hunting for missing packages halfway through a lesson. In a real project you'd normally put these in a `requirements.txt`, not in the notebook — but for a *teaching* notebook, having them here means you can see exactly what powers each technique.


In [ ]:
# ── Install once, then comment this cell back out ───────────────────────────
# The %pip magic installs into the SAME Python environment the notebook is
# using (safer than plain "pip install" in some setups, e.g. inside conda envs).

# %pip install pandas openpyxl                     # local files: CSV/Excel
# %pip install datasets scikit-learn nltk           # public datasets
# %pip install requests beautifulsoup4 lxml          # web scraping
# %pip install python-docx pdfplumber PyPDF2         # text/PDF extraction
# %pip install pytesseract pillow                    # OCR (also needs the
#                                                     # Tesseract binary, see Part 6)
# %pip install openai-whisper SpeechRecognition       # audio transcription
# %pip install moviepy                                # video -> audio extraction
# %pip install tenacity                                # retry logic (Part 10)

print("When you're ready, uncomment the lines above and run this cell once.")


## Part 1 — Loading Local Files (CSV, TSV, JSON, Excel)

This is the "hello world" of data acquisition, but it's worth doing properly because the habits you build here (explicit encodings, checking shape, handling missing values) will save you hours of debugging later on messier sources.


In [ ]:
import pandas as pd

# ── Reading a CSV file ───────────────────────────────────────────────────────
# pandas.read_csv() is the workhorse. Key parameters worth knowing on day one:
#   filepath_or_buffer : path, or even a URL string
#   sep                : delimiter; default is "," — use "\t" for TSV files
#   encoding           : "utf-8" is the safe default; try "latin-1" if you get
#                         a UnicodeDecodeError on older/exported files
#   dtype              : force column types (prevents pandas guessing wrong,
#                         e.g. turning a zip code "007" into the integer 7)
#   nrows              : read only the first N rows — great for peeking at a
#                         huge file before loading it fully

# df = pd.read_csv(
#     "data/reviews.csv",
#     sep=",",
#     encoding="utf-8",
#     dtype={"review_id": str},   # keep IDs as strings, not numbers
# )

# ── Always sanity-check right after loading ─────────────────────────────────
# df.shape        -> (rows, columns), catches "did I load an empty file?"
# df.head()       -> eyeball the first rows
# df.info()       -> dtypes + non-null counts, catches missing-value surprises
# df.isna().sum() -> exact count of missing values per column

# Demonstration with an in-memory example so this cell runs with zero setup:
from io import StringIO
sample_csv = StringIO("id,text,label\n1,I loved this movie,positive\n2,Waste of time,negative\n")
df = pd.read_csv(sample_csv)
print(df.shape)
df.head()


In [ ]:
# 🔀 Alternatives for reading tabular text data
#
# | Tool                | Use when...                                              |
# |----------------------|-----------------------------------------------------------|
# | pandas.read_csv       | Default choice; rich dtype/NA handling, joins easy later  |
# | Python's csv module   | You need a streaming, row-by-row reader for a HUGE file   |
# |                        | that won't fit in memory (avoids loading it all at once)  |
# | polars.read_csv       | Same files, but 5-10x faster on multi-GB CSVs (Rust-based)|
# | dask.dataframe        | File is too big for RAM even with polars; needs chunking  |

import csv

# The standard-library csv module streams row-by-row instead of loading the
# whole file into memory — useful for a NLP corpus file that's 20GB+.
sample_csv.seek(0)  # reset our in-memory file back to the start
reader = csv.DictReader(sample_csv)
for row in reader:
    # row is a plain dict: {'id': '1', 'text': 'I loved this movie', ...}
    print(row)


In [ ]:
# ── JSON and JSON Lines (.jsonl) ─────────────────────────────────────────────
# JSON Lines (one JSON object per line) is EXTREMELY common for NLP corpora
# and API dumps — e.g. Hugging Face datasets, Common Crawl subsets, chat logs.

import json

jsonl_text = '{"text": "great product", "label": 1}\n{"text": "terrible product", "label": 0}\n'

records = []
for line in jsonl_text.splitlines():
    records.append(json.loads(line))   # json.loads = parse a STRING into a dict
                                        # (json.load would parse a file OBJECT)

df_jsonl = pd.DataFrame(records)
df_jsonl

# 📋 Copy-paste template for reading a real .jsonl file:
# records = []
# with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         line = line.strip()
#         if line:                       # skip blank lines
#             records.append(json.loads(line))
# df = pd.DataFrame(records)


In [ ]:
# ── Excel files ───────────────────────────────────────────────────────────
# Needs the "openpyxl" engine (installed above) for modern .xlsx files.

# df_excel = pd.read_excel(
#     "data/survey_responses.xlsx",
#     sheet_name="Sheet1",   # or sheet_name=0 for the first sheet,
#                            # or sheet_name=None to load ALL sheets as a dict
#     engine="openpyxl",
# )

print("Excel loading pattern shown in the comment above — needs a real file to run.")


## Part 2 — Loading Public NLP Datasets

Before scraping anything yourself, **check if a ready-made dataset already exists** — it's cleaner, faster, and (usually) properly licensed. Three go-to sources:

1. **Hugging Face `datasets`** — the biggest hub of NLP datasets (sentiment, translation, QA, summarization...).
2. **`scikit-learn`** — small, classic, great for learning (e.g. 20 Newsgroups).
3. **`nltk`** — classic linguistic corpora (Gutenberg books, movie reviews, Reuters news).


In [ ]:
# ── Hugging Face `datasets` ──────────────────────────────────────────────────
# One line downloads (and caches locally) a dataset that's already split into
# train/validation/test and stored efficiently (Apache Arrow format).

# from datasets import load_dataset
#
# imdb = load_dataset("imdb")            # a classic sentiment-analysis dataset
# print(imdb)                            # shows DatasetDict with train/test splits
# print(imdb["train"][0])                # first example: {'text': ..., 'label': ...}
#
# # Convert to pandas if that's what the rest of your pipeline expects:
# df_imdb = imdb["train"].to_pandas()

print("Uncomment above once you have internet access — load_dataset('imdb') "
      "downloads ~25k labeled movie reviews in one call.")


In [ ]:
# ── scikit-learn's built-in text dataset ─────────────────────────────────────
# Great for FIRST experiments because it needs no download step beyond
# scikit-learn itself, and it's small enough to iterate on quickly.

from sklearn.datasets import fetch_20newsgroups

# newsgroups = fetch_20newsgroups(subset="train", categories=["sci.space", "rec.autos"])
# print(newsgroups.target_names)          # the label names
# print(newsgroups.data[0][:300])         # first article, first 300 characters

print("fetch_20newsgroups() needs internet the first time (it caches after) — "
      "structure shown above.")


In [ ]:
# ── NLTK corpora ─────────────────────────────────────────────────────────────
# NLTK ships "downloaders" for dozens of classic corpora used throughout NLP
# education: movie_reviews (sentiment), gutenberg (full books), reuters (news).

import nltk

# nltk.download("movie_reviews")   # run once; caches to ~/nltk_data
# from nltk.corpus import movie_reviews
#
# fileids = movie_reviews.fileids()                  # list of document IDs
# first_review_words = movie_reviews.words(fileids[0])  # tokenized already!
# first_review_raw = movie_reviews.raw(fileids[0])      # or get raw text

print("nltk.download(...) needs internet the first run; afterwards it's fully offline.")

# 🔀 Alternatives worth knowing about:
# - torchtext.datasets   -> if you're already in a PyTorch pipeline
# - tensorflow_datasets  -> if you're in a TensorFlow pipeline
# - Kaggle API (kaggle datasets download -d <dataset>) -> huge community-uploaded sets


## Part 3 — Web Scraping

Scraping is what you reach for when the data you need has **no API and no ready-made dataset**. Two rules before you write a single line of scraping code:

1. **Check `https://<site>/robots.txt`** — it tells you which paths the site owner allows/disallows for bots.
2. **Check the site's Terms of Service** for automated access — scraping isn't automatically legal just because the data is publicly visible.

We'll use `requests` (to download the HTML) + `BeautifulSoup` (to parse it).


In [ ]:
import requests
from bs4 import BeautifulSoup

# ── Step 1: check robots.txt programmatically ────────────────────────────────
from urllib.robotparser import RobotFileParser

def is_scraping_allowed(url, user_agent="*"):
    """Return True if robots.txt allows fetching `url` for `user_agent`."""
    rp = RobotFileParser()
    root = "/".join(url.split("/")[:3])   # e.g. "https://example.com"
    rp.set_url(root + "/robots.txt")
    rp.read()
    return rp.can_fetch(user_agent, url)

# print(is_scraping_allowed("https://example.com/articles"))
print("is_scraping_allowed() defined — always check this before scraping a new site.")


In [ ]:
# ── Step 2: fetch and parse a single page ────────────────────────────────────

def fetch_page_text(url, timeout=10):
    """
    Download `url` and return its main visible text.
    This is a TEMPLATE — the CSS selectors in the last section will always
    need adjusting per site, since every site's HTML structure is different.
    """
    headers = {
        # Identify yourself honestly — some sites block the default
        # python-requests user agent outright.
        "User-Agent": "Mozilla/5.0 (compatible; MyNLPBot/1.0; +mailto:you@example.com)"
    }
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()          # raises an exception on 4xx/5xx status codes

    soup = BeautifulSoup(response.text, "lxml")   # "lxml" parser: fast + lenient

    # Remove elements that are never "content" (script/style tags, nav menus...)
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    # .get_text() flattens everything into one string;
    # separator="\n" keeps paragraph breaks instead of squashing text together
    text = soup.get_text(separator="\n", strip=True)
    return text

# text = fetch_page_text("https://example.com")
# print(text[:500])
print("fetch_page_text() defined — this network call needs a live internet connection.")


In [ ]:
# ── Step 3: targeted extraction with CSS selectors ───────────────────────────
# Grabbing ALL text (like above) is noisy. In practice you inspect the page
# (browser DevTools -> right-click -> Inspect) and target specific elements.

def scrape_articles(url):
    """
    Example: scrape a hypothetical blog listing page where every article
    is inside <article class="post"> ... <h2 class="title">...</h2>
    <div class="content">...</div> ... </article>
    Adjust the selectors to match the REAL site's markup.
    """
    headers = {"User-Agent": "Mozilla/5.0 (compatible; MyNLPBot/1.0)"}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")

    articles = []
    for post in soup.select("article.post"):          # CSS selector: tag.class
        title_tag = post.select_one("h2.title")
        content_tag = post.select_one("div.content")
        articles.append({
            "title": title_tag.get_text(strip=True) if title_tag else None,
            "content": content_tag.get_text(strip=True) if content_tag else None,
        })
    return articles

print("scrape_articles() defined — a real template you'll adapt selector-by-selector per site.")


In [ ]:
# ── Step 4: pagination (most real scrapes span many pages) ──────────────────
import time

def scrape_paginated(base_url, num_pages, delay_seconds=1.5):
    """
    📋 COPY-PASTE TEMPLATE
    Loops over page numbers, scraping each one, and being polite about it.
    `delay_seconds` between requests avoids hammering the server — being a
    good citizen also lowers your chance of getting IP-banned.
    """
    all_articles = []
    for page_num in range(1, num_pages + 1):
        url = f"{base_url}?page={page_num}"
        try:
            page_articles = scrape_articles(url)
            all_articles.extend(page_articles)
        except requests.exceptions.RequestException as e:
            # Network errors happen (timeouts, DNS issues, 500s) — log and
            # keep going rather than letting one bad page kill the whole run.
            print(f"Failed on page {page_num}: {e}")
        time.sleep(delay_seconds)     # be polite; also avoids rate-limit bans
    return all_articles

print("scrape_paginated() defined.")

# 🔀 Alternatives for web scraping
# | Tool                  | Use when...                                                |
# |-------------------------|-----------------------------------------------------------|
# | requests + BeautifulSoup| Static HTML pages (what we did above) — simplest & fastest|
# | Scrapy                  | Large, structured scraping projects (built-in scheduler,  |
# |                          | retries, pipelines, respects robots.txt automatically)    |
# | Selenium / Playwright    | Pages that render content with JavaScript (SPAs) — these  |
# |                          | tools drive a real browser so JS actually executes         |
# | newspaper3k / trafilatura| Purpose-built for extracting "just the article text" from |
# |                          | news pages, skipping ads/comments automatically             |


## Part 4 — Fetching Data from APIs

APIs are the **cleanest** data source: structured JSON, documented fields, usually a stable contract. The main new skills are **authentication**, **pagination**, and **rate-limit handling**.


In [ ]:
# ── A public API that needs no auth: Wikipedia's REST API ───────────────────
# Good first example because you can run it with zero API-key setup.

def fetch_wikipedia_summary(title):
    """Fetch a plain-text summary of a Wikipedia article by title."""
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()          # .json() parses the response body for you
    return data.get("extract")      # the plain-text summary field

# summary = fetch_wikipedia_summary("Natural_language_processing")
# print(summary)
print("fetch_wikipedia_summary() defined — needs live internet to actually call the API.")


In [ ]:
# ── An authenticated API pattern (API-key in header) ─────────────────────────
# Most production APIs (OpenAI, Twitter/X, NewsAPI, Reddit...) need a key.
# NEVER hardcode a real key in a notebook you might share/commit to git —
# load it from an environment variable instead.

import os

API_KEY = os.environ.get("MY_API_KEY", "")   # set this in your shell/`.env` file, not here

def fetch_from_authenticated_api(endpoint, params=None):
    """
    📋 COPY-PASTE TEMPLATE for a typical bearer-token REST API call.
    """
    headers = {"Authorization": f"Bearer {API_KEY}"}
    response = requests.get(endpoint, headers=headers, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

print("Set MY_API_KEY as an environment variable, never paste it directly into a notebook.")


In [ ]:
# ── Handling pagination + rate limits in one reusable function ──────────────

def fetch_all_pages(endpoint, headers=None, params=None, max_pages=10, delay=1.0):
    """
    📋 COPY-PASTE TEMPLATE
    Many APIs return data in pages with a "next" cursor/URL in the response.
    This loop keeps calling until there's no more "next" page or max_pages
    is hit (a safety cap so a bug can't loop forever).
    """
    all_results = []
    url = endpoint
    params = dict(params or {})

    for _ in range(max_pages):
        response = requests.get(url, headers=headers, params=params, timeout=10)

        if response.status_code == 429:
            # 429 = "Too Many Requests" — the API is rate-limiting you.
            # Respect the Retry-After header if present, else back off a fixed amount.
            wait = int(response.headers.get("Retry-After", 5))
            print(f"Rate limited, sleeping {wait}s...")
            time.sleep(wait)
            continue   # retry the SAME request after waiting

        response.raise_for_status()
        payload = response.json()

        all_results.extend(payload.get("results", []))

        next_url = payload.get("next")   # convention varies by API! Sometimes
                                          # it's `next_page`, a cursor token, etc.
                                          # Always check the specific API's docs.
        if not next_url:
            break
        url, params = next_url, None     # the "next" URL usually already has
                                          # its own query params baked in
        time.sleep(delay)

    return all_results

print("fetch_all_pages() defined — the single most reusable function in this notebook.")

# 🔀 Alternatives / extras for API work
# - praw          -> a friendly wrapper specifically for the Reddit API
# - tweepy         -> wrapper for the X (Twitter) API
# - google-api-python-client -> Google APIs (YouTube comments, etc.)
# - GraphQL APIs use POST requests with a query string in the body instead of
#   GET + query params — the auth/rate-limit patterns above still apply


## Part 5 — Extracting Text from Files (.txt, .docx, .pdf)

Corpora often arrive as documents rather than tables. Plain `.txt` is trivial; `.docx` and `.pdf` need dedicated libraries because they're really zipped XML / binary formats under the hood.


In [ ]:
# ── Plain text files ──────────────────────────────────────────────────────
# with-statement guarantees the file handle is closed even if an error occurs.

# with open("data/document.txt", "r", encoding="utf-8") as f:
#     text = f.read()          # whole file as one string
#     # OR: lines = f.readlines()   # list of lines, if you need line-by-line

print("Plain-text reading pattern shown above.")


In [ ]:
# ── Word documents (.docx) ───────────────────────────────────────────────────
# from docx import Document   # pip package is "python-docx", import name is "docx"
#
# doc = Document("data/report.docx")
# paragraphs = [p.text for p in doc.paragraphs]
# full_text = "\n".join(paragraphs)
#
# # Tables need separate handling — python-docx doesn't merge them into .paragraphs
# for table in doc.tables:
#     for row in table.rows:
#         cells_text = [cell.text for cell in row.cells]
#         print(cells_text)

print(".docx extraction pattern shown above — needs a real .docx file to run.")


In [ ]:
# ── PDF files: two common libraries, compared ────────────────────────────────
#
# | Library     | Strength                                            | Weakness            |
# |--------------|------------------------------------------------------|----------------------|
# | PyPDF2       | Lightweight, no extra deps, fine for clean text PDFs | Struggles with tables/columns/layout |
# | pdfplumber   | Much better layout & TABLE extraction                | Slightly slower      |
#
# Rule of thumb: start with pdfplumber; drop to PyPDF2 only if you need
# something ultra-lightweight and the PDFs are simple.

# --- PyPDF2 ---
# import PyPDF2
# with open("data/paper.pdf", "rb") as f:            # "rb" = read binary, required for PDFs
#     reader = PyPDF2.PdfReader(f)
#     text = ""
#     for page in reader.pages:
#         text += page.extract_text() or ""          # extract_text() can return None
#                                                      # on image-only pages — hence `or ""`

# --- pdfplumber (recommended default) ---
# import pdfplumber
# with pdfplumber.open("data/paper.pdf") as pdf:
#     full_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
#     # pdfplumber can also pull tables directly:
#     first_page_tables = pdf.pages[0].extract_tables()

print("Both PDF extraction patterns shown above — needs a real .pdf file to run.")

# ⚠️ Important: if a PDF is a SCAN (a photo of a page, not real text), both
# libraries above will return empty/garbage text. That's when you need OCR —
# see Part 6, and note pdf2image + pytesseract is the standard combo for
# "scanned PDF -> text".


## Part 6 — Extracting Text from Images (OCR)

OCR = Optical Character Recognition: turning pixels into text. Needed for scanned documents, screenshots, memes-as-data, receipts, etc.

**One-time system setup**: `pytesseract` is a Python wrapper — it needs the actual **Tesseract OCR engine** installed on your machine separately:
- macOS: `brew install tesseract`
- Ubuntu/Debian: `sudo apt-get install tesseract-ocr`
- Windows: install the binary from the Tesseract GitHub releases page


In [ ]:
# from PIL import Image
# import pytesseract
#
# image = Image.open("data/scanned_page.png")
# text = pytesseract.image_to_string(image)
# print(text)

print("Basic OCR pattern shown above.")


In [ ]:
# ── Preprocessing usually improves OCR accuracy a LOT ────────────────────────
# Tesseract works best on high-contrast, de-noised, deskewed black-and-white
# images. A quick preprocessing pipeline:

# from PIL import Image, ImageOps, ImageFilter
#
# def preprocess_for_ocr(image_path):
#     img = Image.open(image_path)
#     img = ImageOps.grayscale(img)                 # drop color info -> higher contrast
#     img = img.filter(ImageFilter.MedianFilter())   # reduce speckle noise
#     # Simple binarization: pixels below 140 -> black, above -> white
#     img = img.point(lambda pixel: 0 if pixel < 140 else 255, mode="1")
#     return img
#
# clean_image = preprocess_for_ocr("data/scanned_page.png")
# text = pytesseract.image_to_string(clean_image)

print("Preprocessing pipeline shown above — this single step often fixes half of all OCR errors.")

# 🔀 Alternatives for OCR
# | Tool                    | Use when...                                          |
# |---------------------------|-----------------------------------------------------|
# | pytesseract (Tesseract)   | Free, offline, good default for clean scans           |
# | easyocr                   | Better out-of-the-box accuracy, supports 80+ languages,|
# |                            | GPU-accelerated, but heavier install                  |
# | Google Cloud Vision API    | Best raw accuracy, handles messy/handwritten text,     |
# |                            | but costs money and needs internet + credentials       |
# | AWS Textract               | Best for structured docs: forms, tables, invoices      |


## Part 7 — Extracting Text from Audio (Transcription)

Podcasts, interviews, call-center recordings, voice notes — all become NLP-ready once transcribed.


In [ ]:
# ── OpenAI's Whisper (open-source, runs locally, no API key needed) ─────────
# Currently the most accurate general-purpose option you can run for free.

# import whisper
#
# model = whisper.load_model("base")   # size options: tiny, base, small, medium,
#                                       # large — bigger = more accurate & slower.
#                                       # "base" is a good default to start with.
# result = model.transcribe("data/interview.mp3")
# print(result["text"])                # the full transcript
# print(result["segments"])            # timestamped chunks, useful for subtitles

print("Whisper transcription pattern shown above.")


In [ ]:
# ── SpeechRecognition library (wraps several cloud/offline engines) ─────────
# Good when you specifically want Google's free web speech API for quick demos,
# or want to swap between multiple recognition backends easily.

# import speech_recognition as sr
#
# recognizer = sr.Recognizer()
# with sr.AudioFile("data/interview.wav") as source:   # needs .wav, not .mp3 directly
#     audio_data = recognizer.record(source)
# text = recognizer.recognize_google(audio_data)        # free, but rate-limited & online-only
# print(text)

print("SpeechRecognition pattern shown above.")

# 🔀 Alternatives for audio transcription
# | Tool                          | Use when...                                    |
# |----------------------------------|-------------------------------------------------|
# | openai-whisper (local)           | Best free accuracy, offline, no per-minute cost |
# | SpeechRecognition + Google API    | Quick prototypes, short clips, free tier limits |
# | AWS Transcribe / Azure Speech      | Production-scale, need speaker diarization,     |
# |                                    | custom vocab, SLAs — but costs money            |
# | AssemblyAI / Deepgram               | Developer-friendly hosted APIs with extras like  |
# |                                    | sentiment tagging and PII redaction built-in      |


## Part 8 — Extracting Text from Video

Video has no direct "transcribe" step — the standard pipeline is:
**video → extract audio track → transcribe audio (Part 7) → text**


In [ ]:
# from moviepy.editor import VideoFileClip
#
# def extract_audio_from_video(video_path, audio_out_path="extracted_audio.wav"):
#     clip = VideoFileClip(video_path)
#     clip.audio.write_audiofile(audio_out_path)   # writes a standalone .wav file
#     clip.close()                                 # release the file handle
#     return audio_out_path
#
# audio_path = extract_audio_from_video("data/lecture.mp4")
#
# # Then reuse the Whisper code from Part 7:
# # import whisper
# # model = whisper.load_model("base")
# # result = model.transcribe(audio_path)
# # print(result["text"])

print("Video -> audio -> transcript pipeline shown above.")

# 🔀 Alternatives for video
# - ffmpeg directly (moviepy actually wraps ffmpeg under the hood) — use raw
#   ffmpeg commands if you want more control or you're not in Python at all:
#   ffmpeg -i lecture.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 lecture.wav
# - YouTube specifically: `youtube-transcript-api` fetches EXISTING captions
#   directly, skipping audio extraction and transcription entirely (fastest
#   path when captions already exist)


## Part 9 — Consolidating Everything into One Corpus

Whatever the source, you almost always want to land on **one consistent format** before moving to the next NLP stage (cleaning/tokenization). A simple, flexible choice: a list of dicts with at least `text` and `source`, saved as JSON Lines.


In [ ]:
# 📋 COPY-PASTE TEMPLATE — a unified record schema across ALL sources above

def make_record(text, source, **extra_metadata):
    """
    Wrap any piece of acquired text into one consistent dict shape,
    regardless of whether it came from a CSV row, a scraped page, an API
    response, a PDF, an OCR'd image, or a transcript.
    """
    record = {
        "text": text,
        "source": source,       # e.g. "csv", "web_scrape", "api", "pdf", "ocr", "audio"
    }
    record.update(extra_metadata)   # e.g. url=..., page_number=..., timestamp=...
    return record

corpus = [
    make_record("I loved this movie", source="csv", label="positive"),
    make_record("Full text of a scraped article...", source="web_scrape",
                 url="https://example.com/article"),
]

corpus_df = pd.DataFrame(corpus)
corpus_df


In [ ]:
# ── Saving the consolidated corpus ────────────────────────────────────────

# As JSON Lines (recommended for NLP corpora — streams well, one record per line):
# with open("data/my_corpus.jsonl", "w", encoding="utf-8") as f:
#     for record in corpus:
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

# As CSV (fine for smaller, purely tabular corpora):
# corpus_df.to_csv("data/my_corpus.csv", index=False, encoding="utf-8")

print("Saving patterns shown above — pick JSONL for large/nested data, CSV for simple tabular data.")


## Part 10 — Making This Production-Ready

The techniques above work in a notebook. Three upgrades make them safe to run unattended in a real pipeline:

1. **Retries with backoff** — networks fail transiently; retry a few times before giving up.
2. **Caching** — never re-download/re-transcribe something you already have.
3. **Logging instead of `print`** — so failures are traceable in production, not lost in stdout.

Also, always keep the **legal/ethical checklist** in mind: robots.txt, ToS, rate limits, and — for anything involving personal data — privacy law (GDPR/CCPA) compliance before you store or use it.


In [ ]:
import logging
from tenacity import retry, stop_after_attempt, wait_exponential

# ── Logging setup (replaces scattered print statements) ─────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("data_acquisition")

# ── Retry decorator: retries up to 3 times, waiting longer each time ────────
# (this "exponential backoff" pattern avoids hammering a struggling server)
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def robust_fetch(url, **kwargs):
    """
    📋 COPY-PASTE TEMPLATE — drop this decorator onto ANY network-calling
    function from this notebook (scraping, API calls) for free retry logic.
    """
    logger.info(f"Fetching {url}")
    response = requests.get(url, timeout=10, **kwargs)
    response.raise_for_status()
    return response

# robust_fetch("https://example.com")
print("robust_fetch() defined — wrap any network call in this pattern for production use.")


In [ ]:
# ── Simple on-disk caching so re-runs don't re-fetch the same URL ───────────
import hashlib
import os

CACHE_DIR = "cache"

def fetch_with_cache(url, fetch_fn):
    """
    📋 COPY-PASTE TEMPLATE
    Hashes the URL to a filename, checks if it's already cached, and only
    calls fetch_fn(url) (your actual network call) on a cache MISS.
    """
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache_key = hashlib.md5(url.encode()).hexdigest()
    cache_path = os.path.join(CACHE_DIR, f"{cache_key}.txt")

    if os.path.exists(cache_path):
        logger.info(f"Cache hit for {url}")
        with open(cache_path, "r", encoding="utf-8") as f:
            return f.read()

    logger.info(f"Cache miss for {url}, fetching...")
    result = fetch_fn(url)
    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(result)
    return result

print("fetch_with_cache() defined — essential once a scrape/API job takes hours to run.")


### Legal & ethical checklist before you scrape/collect anything for real

- [ ] Checked `robots.txt` for the target site
- [ ] Read the site's Terms of Service regarding automated access
- [ ] Set a real, identifiable `User-Agent` header
- [ ] Added delays between requests (rate-limit yourself before they do it for you)
- [ ] If data includes personal information: confirmed a lawful basis for collecting/storing it
- [ ] Checked the dataset/API's license before using it to train or publish anything
- [ ] Stored API keys in environment variables / secrets manager — never in code or notebooks


## Recap & What's Next

You now have working templates for acquiring NLP data from:
**local files → public datasets → the open web → APIs → documents (text/docx/PDF) → images (OCR) → audio → video.**

### Try this before the next lesson (hands-on practice)
1. Pick a topic you're interested in. Find a public dataset for it on Hugging Face and load it.
2. Find one webpage on that topic and scrape its main text using the templates in Part 3 — remember to check `robots.txt` first.
3. Take a screenshot of some text (a tweet, a book page) and OCR it with `pytesseract`.
4. Combine all three into one `corpus.jsonl` using the Part 9 template.

### Next lesson in your NLP mastery path
**Module 2: Data Cleaning & Preprocessing** — handling noisy scraped/OCR'd/transcribed text, normalization, tokenization strategies, and why "clean" means something different for a scraped webpage vs. a transcribed podcast.
